In [3]:
from pathlib import Path
import json, numpy as np, pandas as pd, xgboost as xgb, shap

SEED = 42
np.random.seed(SEED)

ROOT      = Path(r"C:\Fintech-Project\graph_AML_pipeline")
DATA_PROC = ROOT / "data" / "processed"
OUT_DIR   = ROOT / "outputs"
(OUT_DIR / "shap").mkdir(parents=True, exist_ok=True)

def find_one(name):
    hits = list(OUT_DIR.rglob(name))
    assert len(hits) == 1, f"{name}: expected 1 match, found {len(hits)} -> {hits}"
    return hits[0]

A = {n: find_one(n) for n in [
    "m1_baseline.json", "m2_graph_xgb.json",
    "m1_test_probs.parquet", "m2_test_probs.parquet",
    "06_m1_features.json", "07_m2_features.json",
]}
for k, v in A.items():
    print(f"{k:26s} {v.relative_to(ROOT)}")
print(f"\nxgboost {xgb.__version__} | shap {shap.__version__} | pandas {pd.__version__}")

c:\Users\chara\anaconda3\envs\graphaml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


m1_baseline.json           outputs\models\m1_baseline.json
m2_graph_xgb.json          outputs\models\m2_graph_xgb.json
m1_test_probs.parquet      outputs\models\m1_test_probs.parquet
m2_test_probs.parquet      outputs\models\m2_test_probs.parquet
06_m1_features.json        outputs\models\06_m1_features.json
07_m2_features.json        outputs\models\07_m2_features.json

xgboost 2.1.4 | shap 0.46.0 | pandas 2.2.3


In [4]:
m1 = xgb.Booster(); m1.load_model(str(A["m1_baseline.json"]))
m2 = xgb.Booster(); m2.load_model(str(A["m2_graph_xgb.json"]))

TAB_FEATS = json.load(open(A["06_m1_features.json"]))
M2_FEATS  = json.load(open(A["07_m2_features.json"]))
GRAPH_FEATS = [f for f in M2_FEATS if f.startswith(("from_g_", "to_g_"))]

assert len(TAB_FEATS) == 10 and len(M2_FEATS) == 26 and len(GRAPH_FEATS) == 16
assert M2_FEATS[:10] == TAB_FEATS, "shared features must stay in M1 order"
print(f"M1 {len(TAB_FEATS)} features · M2 {len(M2_FEATS)} features · graph family {len(GRAPH_FEATS)}")
print(f"shared features for H4(a): {len(TAB_FEATS)}")

M1 10 features · M2 26 features · graph family 16
shared features for H4(a): 10


In [5]:
df = pd.read_parquet(DATA_PROC / "05_features_full.parquet")
assert len(df) == 5_078_345, "row count does not match the registered artefact"

m1p = pd.read_parquet(A["m1_test_probs.parquet"])
m2p = pd.read_parquet(A["m2_test_probs.parquet"])

# read the flag — never re-derive the wind-down mask
main_idx = m1p.index[m1p["is_main"].values]
assert m1p.index.equals(m2p.index), "M1 and M2 probability files are not on one index"
assert (m1p["is_main"].values == m2p["is_main"].values).all()
assert len(main_idx) == 760_531, f"D1 violated: got {len(main_idx):,}, expected test-main"
assert df.loc[main_idx, "Is Laundering"].sum() == 906

# D1: 10,000 rows sampled from TEST-MAIN, identical indices for both models
sample_idx = pd.Index(main_idx).to_series().sample(10_000, random_state=SEED).index
X_sample = df.loc[sample_idx]

assert sample_idx.isin(main_idx).all(), "sample escaped test-main"
np.save(OUT_DIR / "shap" / "08_test_indices.npy", sample_idx.values)
print(f"sample {len(sample_idx):,} rows · {int(X_sample['Is Laundering'].sum())} illicit "
      f"· drawn from test-main {len(main_idx):,}")

sample 10,000 rows · 8 illicit · drawn from test-main 760,531


In [6]:
train = df[df["split"] == "train"]
n_bg  = 1000
n_pos = max(1, int(n_bg * train["Is Laundering"].mean()))
bg_pos = train[train["Is Laundering"] == 1].sample(n_pos, random_state=SEED)
bg_neg = train[train["Is Laundering"] == 0].sample(n_bg - n_pos, random_state=SEED)
background = pd.concat([bg_pos, bg_neg]).sample(frac=1, random_state=SEED)

assert len(background) == 1000
print(f"background {len(background):,} rows · {int(background['Is Laundering'].sum())} illicit")

background 1,000 rows · 1 illicit


In [7]:
# derive the three time features and encode the categoricals, exactly as notebook 07 did
df["hour"]        = df["Timestamp"].dt.hour
df["day_of_week"] = df["Timestamp"].dt.dayofweek
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

CAT_COLS = ["Receiving Currency", "Payment Currency", "Payment Format"]
for c in CAT_COLS:
    df[c] = df[c].astype("category").cat.codes

# the codes must be identical to notebook 06's, or every SHAP value is misattributed
maps = json.load(open(find_one("06_category_maps.json")))
for c in CAT_COLS:
    expected = {int(k): v for k, v in maps[c].items()}
    actual = dict(enumerate(pd.read_parquet(DATA_PROC / "05_features_full.parquet",
                                            columns=[c])[c].astype("category").cat.categories))
    assert expected == actual, f"category codes differ from notebook 06 for {c}"
    print(f"{c} -> {len(actual)} categories, codes match")

# rebuild both frames on the completed table; same indices, same seed, same rows
X_sample = df.loc[sample_idx]

train  = df[df["split"] == "train"]
n_bg   = 1000
n_pos  = max(1, int(n_bg * train["Is Laundering"].mean()))
bg_pos = train[train["Is Laundering"] == 1].sample(n_pos, random_state=SEED)
bg_neg = train[train["Is Laundering"] == 0].sample(n_bg - n_pos, random_state=SEED)
background = pd.concat([bg_pos, bg_neg]).sample(frac=1, random_state=SEED)

missing = [f for f in M2_FEATS if f not in df.columns]
assert not missing, f"still missing: {missing}"
assert X_sample[M2_FEATS].isna().sum().sum() == 0
assert background[M2_FEATS].isna().sum().sum() == 0
print(f"\nsample {len(X_sample):,} rows · {int(X_sample['Is Laundering'].sum())} illicit")
print(f"background {len(background):,} rows · {int(background['Is Laundering'].sum())} illicit")

Receiving Currency -> 15 categories, codes match
Payment Currency -> 15 categories, codes match
Payment Format -> 7 categories, codes match

sample 10,000 rows · 8 illicit
background 1,000 rows · 1 illicit


In [8]:
import time

thr_m1 = json.load(open(find_one("06_m1_threshold.json")))
thr_m2 = json.load(open(find_one("07_m2_threshold.json")))
n1, n2 = thr_m1["n_trees_used"], thr_m2["n_trees_used"]
assert n1 == 482 and n2 == 430, f"tree counts off-spec: {n1}, {n2}"

# slice to the same trees Notebooks 06/07 used via iteration_range
m1_used, m2_used = m1[:n1], m2[:n2]

# the sliced models must reproduce the probabilities already on disk
p1 = m1_used.predict(xgb.DMatrix(X_sample[TAB_FEATS]))
p2 = m2_used.predict(xgb.DMatrix(X_sample[M2_FEATS]))
d1 = np.abs(p1 - m1p.loc[sample_idx, "m1_prob"].values).max()
d2 = np.abs(p2 - m2p.loc[sample_idx, "m2_prob"].values).max()

print(f"M1 {n1} trees · max prob diff {d1:.3e}")
print(f"M2 {n2} trees · max prob diff {d2:.3e}")
assert d1 < 1e-6 and d2 < 1e-6, "sliced model does not reproduce the scored predictions"
print("\nSHAP will explain exactly the models H1 was decided on")

M1 482 trees · max prob diff 0.000e+00
M2 430 trees · max prob diff 0.000e+00

SHAP will explain exactly the models H1 was decided on


In [9]:
t0 = time.time()
expl_m1 = shap.TreeExplainer(
    m1_used, data=background[TAB_FEATS],
    feature_perturbation="interventional", model_output="raw",
)
shap_m1 = expl_m1.shap_values(X_sample[TAB_FEATS])
print(f"M1 shap {shap_m1.shape} · base {expl_m1.expected_value:.6f} · {time.time()-t0:.1f}s")

100%|===================| 9961/10000 [02:03<00:00]        

M1 shap (10000, 10) · base -6.121312 · 123.7s


In [10]:
t0 = time.time()
expl_m2 = shap.TreeExplainer(
    m2_used, data=background[M2_FEATS],
    feature_perturbation="interventional", model_output="raw",
)
shap_m2 = expl_m2.shap_values(X_sample[M2_FEATS])
print(f"M2 shap {shap_m2.shape} · base {expl_m2.expected_value:.6f} · {time.time()-t0:.1f}s")

100%|===================| 9987/10000 [01:50<00:00]        

M2 shap (10000, 26) · base -6.330573 · 109.5s


In [11]:
for name, expl, sv, bst, feats in [
    ("M1", expl_m1, shap_m1, m1_used, TAB_FEATS),
    ("M2", expl_m2, shap_m2, m2_used, M2_FEATS),
]:
    margin = bst.predict(xgb.DMatrix(X_sample[feats]), output_margin=True)
    recon  = sv.sum(axis=1) + expl.expected_value
    gap    = np.abs(recon - margin).max()
    print(f"{name} additivity · max gap {gap:.3e}")
    assert gap < 1e-3, f"{name} SHAP values do not reconstruct the model output"

np.save(OUT_DIR / "shap" / "08_m1_shap.npy", shap_m1)
np.save(OUT_DIR / "shap" / "08_m2_shap.npy", shap_m2)
json.dump({"m1_base": float(expl_m1.expected_value),
           "m2_base": float(expl_m2.expected_value),
           "n_sample": int(len(sample_idx)), "n_background": 1000,
           "m1_trees": n1, "m2_trees": n2, "seed": SEED,
           "population": "test-main", "perturbation": "interventional"},
          open(OUT_DIR / "shap" / "08_shap_meta.json", "w"), indent=2)
print("\nsaved")

M1 additivity · max gap 3.602e-05
M2 additivity · max gap 3.102e-05

saved


In [12]:
from scipy.stats import spearmanr

mabs_m1 = pd.Series(np.abs(shap_m1).mean(axis=0), index=TAB_FEATS).sort_values(ascending=False)
mabs_m2 = pd.Series(np.abs(shap_m2).mean(axis=0), index=M2_FEATS).sort_values(ascending=False)

def family(f):
    if f.startswith("from_g_"): return "graph_sender"
    if f.startswith("to_g_"):   return "graph_receiver"
    return "transaction"

rank_m2 = pd.DataFrame({"mean_abs_shap": mabs_m2,
                        "family": [family(f) for f in mabs_m2.index],
                        "share_%": 100 * mabs_m2 / mabs_m2.sum()})
rank_m2.insert(0, "rank", range(1, len(rank_m2) + 1))

print("M1 ranking\n", mabs_m1.to_string(), "\n")
print("M2 top 12\n", rank_m2.head(12).to_string())

mabs_m1.to_frame("mean_abs_shap").to_csv(OUT_DIR / "tables" / "08_m1_global_shap.csv")
rank_m2.to_csv(OUT_DIR / "tables" / "08_m2_global_shap.csv")

M1 ranking
 Payment Format        1.245876
From Bank             1.003990
Amount Paid           0.514667
Amount Received       0.375067
To Bank               0.225370
hour                  0.202541
Receiving Currency    0.175651
day_of_week           0.167450
Payment Currency      0.158629
is_weekend            0.039025 

M2 top 12
                      rank  mean_abs_shap          family    share_%
Payment Format          1       1.029871     transaction  17.636690
Amount Paid             2       0.541786     transaction   9.278173
from_g_pagerank         3       0.506927    graph_sender   8.681209
Amount Received         4       0.439388     transaction   7.524579
from_g_out_degree       5       0.370437    graph_sender   6.343782
to_g_in_weighted        6       0.355047  graph_receiver   6.080228
from_g_net_flow         7       0.290415    graph_sender   4.973393
to_g_in_degree          8       0.233106  graph_receiver   3.991980
from_g_in_degree        9       0.210637    graph_sen

In [13]:
graph_mass = mabs_m2[GRAPH_FEATS].sum() / mabs_m2.sum()
top10 = list(rank_m2.head(10).index)
graph_in_top10 = [f for f in top10 if f in GRAPH_FEATS]

print(f"H2 PRIMARY (all {len(X_sample):,} sampled rows)")
print(f"  graph-family mass : {graph_mass:6.2%}   (threshold >= 15%)")
print(f"  graph in top-10   : {len(graph_in_top10)} -> {graph_in_top10}")
print(f"  VERDICT: {'SUPPORTED' if graph_mass >= 0.15 and graph_in_top10 else 'NOT SUPPORTED'}")

# secondary: rows M2 flags at its frozen validation threshold
flag = m2p.loc[sample_idx, "m2_prob"].values >= thr_m2["threshold"]
print(f"\nH2 SECONDARY (model-flagged rows) n = {flag.sum():,}")
if flag.sum() < 1000:
    print("  below the 1,000-row degeneracy floor -> report counts only, no claim")
else:
    mabs_f = pd.Series(np.abs(shap_m2[flag]).mean(axis=0), index=M2_FEATS)
    print(f"  graph-family mass : {mabs_f[GRAPH_FEATS].sum() / mabs_f.sum():6.2%}")

H2 PRIMARY (all 10,000 sampled rows)
  graph-family mass : 52.81%   (threshold >= 15%)
  graph in top-10   : 7 -> ['from_g_pagerank', 'from_g_out_degree', 'to_g_in_weighted', 'from_g_net_flow', 'to_g_in_degree', 'from_g_in_degree', 'to_g_out_weighted']
  VERDICT: SUPPORTED

H2 SECONDARY (model-flagged rows) n = 34
  below the 1,000-row degeneracy floor -> report counts only, no claim


In [14]:
shared = TAB_FEATS  # the 10 features present in both models
rho, pval = spearmanr(mabs_m1[shared].values, mabs_m2[shared].values)

a_pass = rho <= 0.70
b_pass = graph_mass >= 0.15

print(f"H4(a) Spearman rho on {len(shared)} shared features : {rho:.4f}  (p = {pval:.4f})")
print(f"      threshold <= 0.70 -> {'PASS' if a_pass else 'FAIL'}")
print(f"H4(b) graph-family SHAP mass : {graph_mass:.2%}")
print(f"      threshold >= 15%  -> {'PASS' if b_pass else 'FAIL'}")
print(f"\nH4 VERDICT: {'SUPPORTED' if (a_pass and b_pass) else 'NOT SUPPORTED'}  (both parts required)")

pd.DataFrame({"feature": shared,
              "m1_rank": mabs_m1[shared].rank(ascending=False).values,
              "m2_rank": mabs_m2[shared].rank(ascending=False).values,
              "m1_mean_abs": mabs_m1[shared].values,
              "m2_mean_abs": mabs_m2[shared].values}) \
  .to_csv(OUT_DIR / "tables" / "08_h4_shared_ranks.csv", index=False)

json.dump({"h4_rho": float(rho), "h4_p": float(pval), "graph_mass": float(graph_mass),
           "h4a_pass": bool(a_pass), "h4b_pass": bool(b_pass),
           "h4_supported": bool(a_pass and b_pass),
           "h2_supported": bool(b_pass and len(graph_in_top10) > 0),
           "n_shared": len(shared), "population": "test-main", "n_sample": len(sample_idx)},
          open(OUT_DIR / "shap" / "08_hypothesis_verdicts.json", "w"), indent=2)
print("\nverdicts written")

H4(a) Spearman rho on 10 shared features : 0.7818  (p = 0.0075)
      threshold <= 0.70 -> FAIL
H4(b) graph-family SHAP mass : 52.81%
      threshold >= 15%  -> PASS

H4 VERDICT: NOT SUPPORTED  (both parts required)

verdicts written
